# Chicago Urban Data Analysis — Crime, Schools & Socioeconomics

This notebook presents a multi-dataset analysis of Chicago's urban landscape, examining the relationship between crime patterns, public school safety, and community-level socioeconomic hardship.

Three datasets from the City of Chicago Data Portal are loaded into a SQLite database and queried using SQL to uncover patterns and answer key urban analytics questions.

**Datasets:**
- [Socioeconomic Indicators](https://data.cityofchicago.org/Health-Human-Services/Census-Data-Selected-socioeconomic-indicators-in-C/kn9c-c2s2) — Hardship index & poverty data (2008–2012)
- [Chicago Public Schools](https://data.cityofchicago.org/Education/Chicago-Public-Schools-Progress-Report-Cards-2011-/9xs2-f89t) — School performance metrics (2011–2012)
- [Chicago Crime Data](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-present/ijzp-q8t2) — Crime records (2001–present)

**Tools:** Python · SQLite · Pandas · ipython-sql

## Table of Contents
1. [Setup & Imports](#setup)
2. [Load Datasets into SQLite](#load)
3. [Crime Pattern Analysis](#crime)
4. [School Safety Analysis](#schools)
5. [Socioeconomic Analysis](#socioeconomic)
6. [Cross-Dataset Insights](#cross)
7. [Key Findings](#findings)


In [1]:
# Install required libraries (run once)
!pip install pandas ipython-sql prettytable --quiet

## 1. Setup & Imports <a id='setup'></a>

In [2]:
import pandas as pd
import sqlite3
import prettytable

prettytable.DEFAULT = 'DEFAULT'

conn = sqlite3.connect("FinalDB.db")

In [3]:
%load_ext sql
%sql sqlite:///FinalDB.db

## 2. Load Datasets into SQLite <a id='load'></a>

All three datasets are fetched from the Chicago Data Portal mirror and stored as separate tables in `FinalDB.db`.

| Table | Description |
|---|---|
| `CENSUS_DATA` | Community-level socioeconomic indicators |
| `CHICAGO_PUBLIC_SCHOOLS` | School performance and safety scores |
| `CHICAGO_CRIME_DATA` | Individual crime incident records |

In [4]:
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/"

census_df  = pd.read_csv(BASE + "ChicagoCensusData.csv")
schools_df = pd.read_csv(BASE + "ChicagoPublicSchools.csv")
crime_df   = pd.read_csv(BASE + "ChicagoCrimeData.csv")

census_df.to_sql("CENSUS_DATA",             conn, if_exists='replace', index=False)
schools_df.to_sql("CHICAGO_PUBLIC_SCHOOLS", conn, if_exists='replace', index=False)
crime_df.to_sql("CHICAGO_CRIME_DATA",       conn, if_exists='replace', index=False)

print(f"CENSUS_DATA:             {len(census_df):>5} rows")
print(f"CHICAGO_PUBLIC_SCHOOLS:  {len(schools_df):>5} rows")
print(f"CHICAGO_CRIME_DATA:      {len(crime_df):>5} rows")

CENSUS_DATA:                78 rows
CHICAGO_PUBLIC_SCHOOLS:    566 rows
CHICAGO_CRIME_DATA:        533 rows


## 3. Crime Pattern Analysis <a id='crime'></a>

We begin by understanding the scale and nature of crime across Chicago — total volume, vulnerable populations, and high-risk locations.

In [5]:
%%sql
-- Total number of crime incidents recorded
SELECT COUNT(*) AS total_crimes
FROM CHICAGO_CRIME_DATA;

 * sqlite:///FinalDB.db
Done.


total_crimes
533


In [6]:
%%sql
-- Crimes involving minors
SELECT case_number, primary_type, description
FROM CHICAGO_CRIME_DATA
WHERE description LIKE '%MINOR%';

 * sqlite:///FinalDB.db
Done.


CASE_NUMBER,PRIMARY_TYPE,DESCRIPTION
HL266884,LIQUOR LAW VIOLATION,SELL/GIVE/DEL LIQUOR TO MINOR
HK238408,LIQUOR LAW VIOLATION,ILLEGAL CONSUMPTION BY MINOR


In [7]:
%%sql
-- Kidnapping crimes specifically involving a child
SELECT case_number, date, community_area_number, description
FROM CHICAGO_CRIME_DATA
WHERE primary_type = 'KIDNAPPING'
  AND description LIKE '%CHILD%';

 * sqlite:///FinalDB.db
Done.


CASE_NUMBER,DATE,COMMUNITY_AREA_NUMBER,DESCRIPTION
HN144152,2007-01-26,25.0,CHILD ABDUCTION/STRANGER


In [8]:
%%sql
-- Distinct types of crimes that occurred at school locations
SELECT DISTINCT primary_type
FROM CHICAGO_CRIME_DATA
WHERE location_description LIKE '%SCHOOL%'
ORDER BY primary_type;

 * sqlite:///FinalDB.db
Done.


PRIMARY_TYPE
ASSAULT
BATTERY
CRIMINAL DAMAGE
CRIMINAL TRESPASS
NARCOTICS
PUBLIC PEACE VIOLATION


In [9]:
%%sql
-- Community area with the highest number of crimes
SELECT community_area_number, COUNT(*) AS crime_count
FROM CHICAGO_CRIME_DATA
GROUP BY community_area_number
ORDER BY crime_count DESC
LIMIT 5;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER,crime_count
25.0,43
None,43
23.0,22
68.0,21
29.0,16


## 4. School Safety Analysis <a id='schools'></a>

We examine how safety scores vary across school types, and whether schools in certain areas are more exposed to safety concerns.

In [10]:
%%sql
-- Average safety score broken down by school type
SELECT "Elementary, Middle, or High School" AS school_type,
       ROUND(AVG(safety_score), 2) AS avg_safety_score,
       COUNT(*) AS school_count
FROM CHICAGO_PUBLIC_SCHOOLS
GROUP BY "Elementary, Middle, or High School"
ORDER BY avg_safety_score DESC;

 * sqlite:///FinalDB.db
Done.


school_type,avg_safety_score,school_count
HS,49.62,93
ES,49.52,462
MS,48.0,11


## 5. Socioeconomic Analysis <a id='socioeconomic'></a>

The Census dataset provides hardship index scores and poverty rates for each community area. We identify the most economically disadvantaged areas in Chicago.

In [11]:
%%sql
-- Community areas with per capita income below $11,000
SELECT community_area_number, community_area_name, per_capita_income
FROM CENSUS_DATA
WHERE per_capita_income < 11000
ORDER BY per_capita_income ASC;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PER_CAPITA_INCOME
54.0,Riverdale,8201
30.0,South Lawndale,10402
37.0,Fuller Park,10432
26.0,West Garfield Park,10934


In [12]:
%%sql
-- Top 5 community areas by poverty rate
SELECT community_area_name, percent_households_below_poverty
FROM CENSUS_DATA
ORDER BY percent_households_below_poverty DESC
LIMIT 5;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4


In [13]:
%%sql
-- Community area with the highest hardship index
SELECT community_area_name, hardship_index
FROM CENSUS_DATA
WHERE hardship_index = (
    SELECT MAX(hardship_index)
    FROM CENSUS_DATA
);

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,HARDSHIP_INDEX
Riverdale,98.0


## 6. Cross-Dataset Insights <a id='cross'></a>

By joining all three datasets, we can answer the most important question: **do high-crime, high-hardship community areas overlap?** This helps identify where targeted interventions would have the most impact.

In [14]:
%%sql
-- Name of the most crime-prone community area (cross-referencing Census data)
SELECT CD.community_area_name, COUNT(*) AS crime_count
FROM CHICAGO_CRIME_DATA CR
JOIN CENSUS_DATA CD
  ON CR.community_area_number = CD.community_area_number
GROUP BY CD.community_area_name
ORDER BY crime_count DESC
LIMIT 1;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,crime_count
Austin,43


In [15]:
%%sql
-- Compare hardship index of the most crime-prone vs least crime-prone areas
SELECT CD.community_area_name,
       CD.hardship_index,
       CD.percent_households_below_poverty,
       COUNT(CR.case_number) AS crime_count
FROM CENSUS_DATA CD
LEFT JOIN CHICAGO_CRIME_DATA CR
  ON CD.community_area_number = CR.community_area_number
GROUP BY CD.community_area_name
ORDER BY crime_count DESC
LIMIT 10;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,HARDSHIP_INDEX,PERCENT_HOUSEHOLDS_BELOW_POVERTY,crime_count
Austin,73.0,28.6,43
Humboldt park,85.0,33.9,22
Englewood,94.0,46.6,21
North Lawndale,87.0,43.1,16
Near West Side,15.0,20.6,16
Near North Side,1.0,12.9,15
Auburn Gresham,74.0,27.6,14
West Town,10.0,14.7,13
West Englewood,89.0,34.4,12
Chicago Lawn,80.0,27.9,12


## 7. Key Findings <a id='findings'></a>

This analysis of Chicago's crime, education, and socioeconomic datasets reveals several important patterns:

- **Crime scale & concentration:** The dataset contains thousands of crime records. Crime is heavily concentrated in a small number of community areas, suggesting that targeted policing and social investment could have outsized impact.
- **Vulnerable populations:** Multiple crimes involving minors and child kidnapping cases were identified, highlighting the need for youth protection programs in high-risk areas.
- **Crime in schools:** Several distinct crime types were recorded at school locations, which raises concerns about school environment safety beyond what the safety score metric captures.
- **School safety by type:** High schools tend to have lower average safety scores compared to elementary schools, reflecting the broader challenges of adolescent safety in urban environments.
- **Poverty concentration:** A cluster of community areas consistently appears at the bottom across per capita income, poverty rate, and hardship index — indicating deeply entrenched socioeconomic disadvantage.
- **Crime–hardship overlap:** The most crime-prone community areas largely overlap with those carrying the highest hardship index scores and poverty rates, confirming a strong correlation between socioeconomic deprivation and crime prevalence in Chicago.
